<img src="https://github.com/milena-seismo/advanced-topic_deep-learning/blob/main/session-2-practice/06-plant-disease-image-classifier/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

## Project: Deep Learning - Building an Image Classifier for Detecting Plant Diseases

---

## The Challenge

You've been hired as a data scientist for an agriculture tech company.

The company aims to help farmers identify diseases in their crops early on, to reduce losses and improve crop yields.

Your task is to **build an image classification model** to detect plant diseases from leaf images.

Your model should classify images into multiple categories, such as healthy leaves or specific diseases (e.g., "Powdery Mildew," "Rust," "Leaf Spot"). This will help farmers take targeted action to treat their crops.

To accomplish this, you'll use a publicly available data set of plant leaf images, the **PlantVillage data set**, and `PyTorch` for building the classification model.

## The Data

You should use the **PlantVillage Data Set**.  This contains thousands of labeled images of healthy and diseased plant leaves, together with disease classification labels.

The data set is freely available and can be accessed or downloaded from [Kaggle](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage).

The academic paper behind the data set is available [here](https://www.frontiersin.org/journals/artificial-intelligence/articles/10.3389/frai.2022.868926/full).


## Deliverables

* A **Jupyter notebook** containing all the code to process your data and build, train and test your model. You can use this notebook as a starting point.

* A **5 minute presentation**, intended for a non technical audience, that outlines and interprets your results

## Instructions

Here is a suggested set of steps to follow:

1. **Set Up Your Notebook** Import the libraries you'll need and get the data set from [Kaggle](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage). It's already split into training and validation sets.

2. **Data Preprocessing and Transformation:**
    - Resize the image size to 224x224 pixels.
    - Normalize the image data.
    - Apply data augmentations like random rotations or flips to improve generalization.

<br />

3. **Initialise (Build) the Model:** Use a pre-trained model (e.g., **ResNet18**) from `torchvision.models` and fine-tune it for the plant disease classification task.

4. **Train the Model:** Train the model on the training data set.
Use a suitable loss function (e.g., cross-entropy) and optimizer (e.g., Adam).

5. **Evaluate the Model:** Evaluate the model on the validation set. Report the overall model accuracy.

6. **Conclusions and Recommendations:** Is your model performance good enough for deployment in the real world? Is your model suitable for detecting some diseases but not others?

## Best practice

Your code should be clean and well commented.

Remember to include text cells in this notebook which explain what each code block is doing, and justify any important choices you've made (e.g. model architecture).

---
# Starter code

Fill in the gaps, edit and add to this starter code as you see fit to complete the task.

If you want more of a challenge, you can also ignore the starter code and write everything from scratch!

### 1. Set Up Your Notebook

#### Import libraries

You must use `pytorch` for this task. Import any modules you'll need below:

In [ ]:
# import modules here

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

#### Get the data

You can either download it to your machine and train a model there, or (recommended) connect the Kaggle data with a Google Colab notebook and train the model in the cloud, using a free GPU.  This will be _much_ quicker than running it locally on your machine.

**a. To use Google Colab**:
1. [Open Colab](https://colab.research.google.com/)
2. File -> Upload notebook
3. Upload this notebook
4. Switch to using a GPU **Change runtime type** and select `T4 GPU`

Get the data:
1. [Go to the data on Kaggle](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage?resource=download)
2. Click on Download, select KaggleHub* and copy the code
3. Paste that code into this notebook and run it

*KaggleHub is a Python package that provides a simple API to access Kaggle resources anywhere, including Colab.


Want to know more about Colab?  See [Google’s Colab site](https://colab.google/) for more details or have a look at the [FAQs](https://research.google.com/colaboratory/faq.html).

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohitsingh1804/plantvillage")

print("Path to dataset files:", path)

**b. To use your local machine**:

Get the data:
1. [Go to the data on Kaggle](https://www.kaggle.com/datasets/mohitsingh1804/plantvillage?resource=download)
2. Click on Download, select **Download dataset as zip**

#### Define your data directories.

These directories should contain the **PlantVillage data set**, which is already split into subfolders for training and validation.

Each subfolder should have class-wise folders (e.g., `__healthy`, `__Powdery_mildew`) with corresponding images.  (You can easily see this if you have downloaded the data to your local machine.)

In [ ]:
data_dir = f"{path}/PlantVillage" # Use this if you are running the notebook online via Colab
# data_dir = "./data" # Use this if you are running the notebook from your local machine (and change the location accordingly)
train_dir = f"{data_dir}/train"
val_dir = f"{data_dir}/val"

Your notebook and data are now ready to use.

### 2. Data Preprocessing and Transformation

* Resize all images to 224x224 pixels to match the input size expected by the **ResNet18** model that we'll be using.

* Apply random horizontal flipping to augment the data. Adding random horizontal flips helps generalize the model to real-world variations, such as flipped leaf images.

* Convert the images to PyTorch tensors.

* Normalize the image pixel values to match the pre-trained **ResNet**'s expected input distribution.

Hint: You can load the images and perform all these transformations easily with the `transform` and `ImageFolder` functions in `pytorch`.

In [ ]:
# Set parameters for transformation

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Ensures consistent input size for the CNN
    transforms.RandomHorizontalFlip(),  # Augments data by creating flipped versions of images
    transforms.ToTensor(),  # Converts image to tensor format
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Standard normalization for pre-trained models
])

# Load data sets using ImageFolder
# - ImageFolder automatically labels images based on folder structure.
# - Use the defined transformations for consistency and augmentation.

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)


When training deep learning models, **batching** and **shuffling** help with computational efficiency and generalization.

#### Batching

Batching refers to dividing the data set into smaller subsets (or "batches") instead of processing the entire data set at once. This improves computational efficiency and training speed.

For example, if the data set has 1,000 samples and the batch size is 32, then:

- **1st Batch**: Samples 1–32
- **2nd Batch**: Samples 33–64
- ...
- **Last Batch (number 32)**: Samples 993–1,000

Each batch is passed through the model independently. After processing a batch, the gradients are computed and the model weights are updated. This process is repeated for every batch in an **epoch** (one complete iteration over the entire data set).

In `pytorch`, the `DataLoader` class automatically divides the data set into batches. The batch size (e.g., batch_size=32) specifies how many samples are in each batch.

#### Shuffling

Shuffling refers to randomly reordering the data set at the start of each epoch. This ensures that the model does not learn any unintended patterns or biases caused by the order of the data.

When `shuffle=True` is specified in the `DataLoader` class, the data set is randomly reordered before creating batches.

Use the `DataLoader` class from `pytorch` to:
* Divide the training and validation sets into batches of 32 images
* Shuffle the training set only

In [ ]:
# Set Dataloaders for batching and shuffling
# DataLoader splits the data into batches and shuffles it for efficient training

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Shuffle ensures diverse batches
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)  # No shuffle for validation set

### 3. Initialise the Model

Now our data is ready, it's time to initialise, or build, our model.

We'll be using a pre-trained **ResNet18** model. **ResNet18** is a popular convolutional neural network (CNN) with 18 layers, pre-trained on [ImageNet](https://www.image-net.org/).  (For more information on ImageNet, see [here](https://en.wikipedia.org/wiki/ImageNet).)

Pytorch provides a [number of classification models](https://docs.pytorch.org/vision/main/models.html#classification) for you to explore and use.  You can use them with or without pre-trained weights.

Pre-trained models save time and leverage transfer learning, making them ideal for image classification tasks with limited data sets.

In [ ]:
model = models.resnet18(pretrained=True)  # Load ResNet18 with pre-trained weights

`ResNet18`'s original output layer has 1000 classes (corresponding to the 1000 classes in ImageNet).

You should replace the output layer with a layer matching the number of classes in **PlantVillage**.

In [ ]:
# fc refers to the final fully connected (FC) layer - or output layer - of a pre-trained model, such as ResNet

model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))  # Adjust to number of classes in the data set

#### Define the loss function

You should use a cross entropy loss function, which is the standard choice for multi-class classification problems.

In [ ]:
criterion = nn.CrossEntropyLoss()  # Measures how far predictions are from true labels

### 4. Train the Model

Train the model for a fixed number of epochs.

For each epoch, iterate through the training data, calculate loss, and update model weights.

In [ ]:
# Set the number of epochs
epochs = 10  # Number of training iterations over the entire data set

In [ ]:
# Define the device that the model will train on. 'cuda' means GPU and CPU means CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # Move model to the device

Define the optimizer.
We will use the Adam optimizer on the final layer of the model .

In [ ]:
# Initialize the optimizer after moving the model to the device
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)  # Optimizes only the final layer (fine-tuning) (not all model parameters)

In [ ]:
# Training loop
for epoch in range(epochs):
    model.train()  # Set model to training mode (enables dropout, batchnorm updates, etc.)
    running_loss = 0.0  # Track cumulative loss for the epoch
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)  # Move data to GPU/CPU

        optimizer.zero_grad()  # Reset gradients (PyTorch accumulates gradients by default)
        outputs = model(inputs)  # Forward pass: Get predictions
        loss = criterion(outputs, labels)  # Compute loss between predictions and true labels
        loss.backward()  # Backward pass: Compute gradients
        optimizer.step()  # Update model weights based on gradients

        running_loss += loss.item()  # Accumulate loss
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")  # Log average loss per epoch

### 5. Evaluate the Model (model validation)

Write a loop to evaluate model performance on the validation set.

Remember, the validation step does not update weights; it only measures the model's classification accuracy on new/unseen data.

Report the overall model accuracy.

In [ ]:
# Validation loop
model.eval()  # Set model to evaluation mode (disables dropout, batchnorm updates, etc.)
correct = 0
total = 0
with torch.no_grad():  # Disable gradient computation for efficiency
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to GPU/CPU
        outputs = model(inputs)  # Forward pass
        _, preds = torch.max(outputs, 1)  # Get class with highest score
        correct += (preds == labels).sum().item()  # Count correctly classified samples
        total += labels.size(0)  # Total number of samples
val_accuracy = correct / total  # Compute validation accuracy
print(f"Validation Accuracy: {val_accuracy:.2f}")

### 6. Conclusions and Recommendations